# Offline Netron Preview: Crashed Stitched-IP Build Intermediates

This notebook runs with a **local Windows Python kernel** (not inside the FINN container), so `netron`'s
local web server is actually reachable from this machine's browser/VS Code -- unlike running it inside the
Docker container (which has no ports published to the host).

Models previewed are copied locally from the crashed `quantEnet_O8_native` stitched-IP build
(`stitched_ip_quantEnet_O8_native_20260729_123711`), covering the pipeline from raw import through
per-node HLS/RTL synthesis (`step_hw_ipgen`, the last step to complete before the container died).

Run each cell below; the Netron viewer will render inline and stay live as long as this kernel is running.


In [ ]:
import os
import threading
import time

import netron
from IPython.display import IFrame

MODELS_DIR = os.path.join(os.getcwd(), "onnx_previews")

_running_ports = set()


def show_in_netron(filename, port):
    """Start a local netron server for the given model file and return an IFrame.

    Runs entirely locally -- the server is on this machine, so it's reachable
    even though the model was produced inside the (unreachable) FINN container.
    """
    path = os.path.join(MODELS_DIR, filename)
    if not os.path.exists(path):
        raise FileNotFoundError(path)

    if port not in _running_ports:
        thread = threading.Thread(
            target=lambda: netron.start(path, address=("localhost", port), browse=False)
        )
        thread.daemon = True
        thread.start()
        _running_ports.add(port)
        time.sleep(1.5)  # give the server a moment to come up

    return IFrame(src=f"http://localhost:{port}/", width="100%", height=500)


## 1. `step_enet_tidy` -- raw imported graph (515 nodes, pre-streamline)

In [ ]:
show_in_netron("step_enet_tidy.onnx", 8081)

## 2. `step_enet_streamline` -- note the Transpose churn (442 nodes)

In [ ]:
show_in_netron("step_enet_streamline.onnx", 8082)

## 3. `step_enet_convert_to_hw` -- converted to FINN HW ops (328 nodes)

In [ ]:
show_in_netron("step_enet_convert_to_hw.onnx", 8083)

## 4. `step_specialize_layers` -- HLS vs RTL backend chosen per node (325 nodes)

In [ ]:
show_in_netron("step_specialize_layers.onnx", 8084)

## 5. `step_hw_ipgen` -- final state before the crash: post per-node HLS/RTL synthesis (325 nodes)

In [ ]:
show_in_netron("step_hw_ipgen.onnx", 8085)